# Kaggle Submission Notebook (BM25 Only)

Minimal OOP notebook for recall-focused BM25 retrieval and Kaggle CSV generation.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import csv
import json
import re

import numpy as np
import pandas as pd


@dataclass
class SubmissionConfig:
    data_dir: Path
    output_path: Path = Path('solutions_SeaFour.csv')
    top_k: int = 100
    doc_columns: tuple[str, ...] = ('title', 'text', 'tags')
    query_columns: tuple[str, ...] = ('title', 'text')


DATA_DIR = Path('/kaggle/input/retrieval-engine-competition')
if not DATA_DIR.exists():
    DATA_DIR = Path('../../data')

config = SubmissionConfig(data_dir=DATA_DIR)


In [ ]:
class BM25SubmissionPipeline:
    token_pattern = re.compile(r'[a-z0-9]+')

    def __init__(self, config: SubmissionConfig):
        self.config = config
        self._docs_df = None
        self._queries_df = None
        self._doc_ids = None
        self._tokenized_corpus = None
        self._query_tokens = None
        self._bm25 = None
        print(f'[init] BM25SubmissionPipeline ready | data_dir={self.config.data_dir} | top_k={self.config.top_k}')

    @staticmethod
    def _value_to_text(value):
        if value is None:
            return ''
        if isinstance(value, (list, tuple)):
            return ' '.join(str(item) for item in value)
        if pd.isna(value):
            return ''
        return str(value)

    def tokenize(self, text):
        text = str(text or '').lower()
        text = re.sub(r'[-_/]', ' ', text)
        return self.token_pattern.findall(text)

    def _build_content(self, df, columns):
        print(f'[content] Building content from columns={columns} for {len(df)} rows')
        prepared = df.copy()
        for column in columns:
            if column not in prepared.columns:
                prepared[column] = ''

        prepared['content'] = prepared[list(columns)].apply(
            lambda row: ' '.join(self._value_to_text(value) for value in row).strip().lower(),
            axis=1,
        )
        prepared['id'] = prepared['id'].astype(str)
        return prepared

    def load_documents(self):
        if self._docs_df is None:
            print('[docs] Loading documents from disk...')
            docs_df = pd.read_json(self.config.data_dir / 'docs.json')
            print(f'[docs] Loaded {len(docs_df)} documents')
            self._docs_df = self._build_content(docs_df, self.config.doc_columns)
            self._doc_ids = self._docs_df['id'].astype(str).to_numpy()
            print('[docs] Documents cached in memory')
        else:
            print(f'[docs] Using cached documents: {len(self._docs_df)} rows')
        return self._docs_df

    def load_queries(self):
        if self._queries_df is None:
            print('[queries] Loading test queries from disk...')
            queries_df = pd.read_json(self.config.data_dir / 'queries_test.json')
            print(f'[queries] Loaded {len(queries_df)} test queries')
            self._queries_df = self._build_content(queries_df, self.config.query_columns)
            print('[queries] Queries cached in memory')
        else:
            print(f'[queries] Using cached queries: {len(self._queries_df)} rows')
        return self._queries_df

    def load_query_tokens(self):
        if self._query_tokens is None:
            print('[queries] Tokenizing queries...')
            queries_df = self.load_queries()
            self._query_tokens = [self.tokenize(text) for text in queries_df['content']]
            print(f'[queries] Cached tokenized queries: {len(self._query_tokens)}')
        else:
            print(f'[queries] Using cached tokenized queries: {len(self._query_tokens)}')
        return self._query_tokens

    def build_index(self):
        if self._bm25 is not None:
            print(f'[index] Using cached BM25 index for {len(self._tokenized_corpus)} documents')
            return self._bm25

        docs_df = self.load_documents()
        print('[index] Tokenizing document corpus...')
        self._tokenized_corpus = [self.tokenize(text) for text in docs_df['content']]
        print(f'[index] Tokenized {len(self._tokenized_corpus)} documents')

        try:
            from rank_bm25 import BM25Plus
        except ImportError:
            import subprocess
            import sys
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rank-bm25'])
            from rank_bm25 import BM25Plus

        print('[index] Building BM25Plus index...')
        self._bm25 = BM25Plus(self._tokenized_corpus)
        print('[index] BM25Plus index ready')
        return self._bm25

    def run_bm25_search(self, docs_df, queries_df):
        top_k = min(self.config.top_k, len(docs_df))
        print(f'[search] Starting BM25 scoring | queries={len(queries_df)} | docs={len(docs_df)} | top_k={top_k}')
        bm25 = self.build_index()
        doc_ids = self._doc_ids
        query_tokens = self.load_query_tokens()

        results = []
        for idx, (row, tokens) in enumerate(zip(queries_df.itertuples(index=False), query_tokens), start=1):
            scores = bm25.get_scores(tokens)
            top_idx = np.argpartition(-scores, top_k - 1)[:top_k]
            top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
            results.append({
                'query_id': str(row.id),
                'relevant_docs': doc_ids[top_idx].tolist(),
            })
            if idx == 1 or idx % 25 == 0 or idx == len(queries_df):
                print(f'[search] Processed {idx}/{len(queries_df)} queries')

        print('[search] BM25 scoring complete')
        return results

    def write_kaggle_submission(self, results):
        sample_submission_path = self.config.data_dir / 'submission.csv'
        print(f'[submission] Writing Kaggle submission for {len(results)} queries...')
        pred_map = {
            str(item['query_id']): [str(doc_id) for doc_id in item['relevant_docs']]
            for item in results
        }

        with open(sample_submission_path, 'r', newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames
            rows = list(reader)

        if fieldnames is None or len(fieldnames) < 2:
            raise ValueError('Invalid sample submission format.')

        id_col = fieldnames[0]
        pred_col = fieldnames[1]
        category_col = fieldnames[2] if len(fieldnames) >= 3 else None

        with open(self.config.output_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

            for row in rows:
                qid = str(row[id_col])
                out_row = {
                    id_col: qid,
                    pred_col: json.dumps(pred_map[qid]),
                }
                if category_col is not None:
                    out_row[category_col] = row.get(category_col, '?') or '?'
                writer.writerow(out_row)
        print(f'[submission] Saved file: {self.config.output_path.resolve()}')

    def generate_submission(self):
        print('[pipeline] Starting submission generation')
        docs_df = self.load_documents()
        queries_df = self.load_queries()
        results = self.run_bm25_search(docs_df, queries_df)
        self.write_kaggle_submission(results)
        print('[pipeline] Submission generation finished')
        return results


In [ ]:
if 'pipeline' not in globals() or pipeline.config != config:
    print('[run] Creating new pipeline instance')
    pipeline = BM25SubmissionPipeline(config)
else:
    print('[run] Reusing existing pipeline instance and caches')
results = pipeline.generate_submission()
print(f'Saved: {config.output_path.resolve()}')


In [ ]:
print('[preview] Loading generated submission preview...')
submission_preview = pd.read_csv(config.output_path)
submission_preview.head()
